# 03 — Two-Stage Reverse (path B)

Reverse Two-Stage (Stage 1 회귀 → Stage 2 분류, weighted MSE) HPO + refit + 후처리.

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_*_data.csv`
- **출력**: `4_output/03_two_stage/reverse/{best_params.json, fold_models.pkl, optuna_*.db, oof|val|test_die.csv, oof|val|test_unit.csv}`
- **PP**: 트리 공통 `PP_FIXED` (strategy_common.md §1) — 1회 사전 적용
- **HPO**: 80 trial, anchor=1차 ts-reverse best (in-sample mode val=0.005709), narrow ±30%, **anchor 첫 trial enqueue**
- **Path B 동작 (각 fold)**:
  1. Stage 1 회귀 (먼저, 모든 die) — target=`y_die_broadcast` (TARGET_TRANSFORM='none', strategy_common §24), sample_weight: `y=0→w0`, `y>0→1.0`, objective ∈ {regression, poisson, tweedie_1.2, tweedie_1.5}
  2. Stage 2 분류 (나중, 모든 die) — X에 Stage 1 reg_pred 보조 feature 추가, scale_pos_weight 탐색
  3. Final die pred = `clf_proba × reg_pred`
  4. Unit pred = `groupby(KEY).mean()`
- **Stage 2 보조 feature 모드**: `inner-OOF` 고정 — outer-train 안에서 inner KFold(K=5, unit-level)로 reg OOF 생성하여 학습-추론 분포 일치
- **후처리**: 집계 8종, position Optuna 50t, zero_clip log space, **π threshold APPLY** (die-level prob → unit mean → threshold)


## 1. 환경 설정 + import

Colab/Local 자동 감지. Colab 사용 시 `GDRIVE_MODELING_ID` 채울 것.

In [ ]:
import os, sys

GDRIVE_CODE_ID         = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'
GDRIVE_DATASET_ID      = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'
GDRIVE_MODELING_ID     = '1Vrn5LBl611rWbag7d09LZH68_lfpu6wP'  # ★ Colab 사용 시 신규 modeling.zip ID 입력

try:
    import google.colab
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if GDRIVE_MODELING_ID and not os.path.exists('/content/project/3_modeling/modules/zit.py'):
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modeling.zip')
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system('unzip -qo /content/modeling.zip -d /content/project/3_modeling')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

# strategy_common.md §22 — 2_preprocessing 직접 import
PP_DIR = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PP_DIR not in sys.path:
    sys.path.insert(0, PP_DIR)

MOD_DIR = os.path.join(PROJECT_ROOT, '3_modeling')
if MOD_DIR not in sys.path:
    sys.path.insert(0, MOD_DIR)

from modules import preprocess, hpo, postprocess
from meta_features import add_meta_features   # 2_preprocessing/meta_features.py — 2026-05-09 결정 반영

import lightgbm as lgb
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.model_selection import KFold

import logging, time
logging.getLogger('lightgbm').setLevel(logging.ERROR)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'optuna v{optuna.__version__}')

## 2. 실험 설정

- `TS_REVERSE_ANCHOR` — 1차 ts-reverse-hpo-001 best HP (val=0.005709, test=0.008412)
- `TS_REVERSE_SEARCH` — narrow_around로 LGBM/w0 자동 산출 + categorical 4종 수동 추가
- categorical: `reg_objective ∈ {regression, poisson, tweedie_1.2, tweedie_1.5}`, `clf_scale_pos_weight ∈ {1.0, 1.5, 2.43, 3.5}` (1차 4종 그대로 탐색)

In [ ]:
# ── 실험 식별 ──
EXP_ID = 'ts-reverse-final-001'
USER   = 'jh'

# ── Optuna 예산 ──
N_TRIALS         = 1
TIMEOUT_SEC      = None  # ★ Colab 타임아웃 대비, 초 단위 (None=무제한)
N_FOLDS          = 5
K_INNER          = 5    # inner-OOF용 inner KFold (unit-level)
N_JOBS           = 7    # ★ 단일 변수 (strategy_common §8)
N_STARTUP_TRIALS = 1

# ── 출력 경로 ──
OUT_DIR = os.path.join(OUTPUT_DIR, '03_two_stage', 'reverse')
os.makedirs(OUT_DIR, exist_ok=True)
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')

# ── y 극단값 clip ──
CLIP_Y_EXTREME = True

# ── PP_FIXED (strategy_common.md §1) ──
PP_FIXED = {
    'missing_threshold':          0.30,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.05,
    'spatial_max_dist':           6.0,
    'post_impute_corr_threshold': 0.96,
    'post_impute_corr_keep_by':   'std',
}

# ── TS_REVERSE_ANCHOR (1차 ts-reverse-hpo-001 best, OOF=0.005495) ──
TS_REVERSE_ANCHOR = {
    'n_estimators':      277,
    'learning_rate':     0.01152,
    'num_leaves':        476,
    'max_depth':         12,
    'min_child_samples': 22,
    'subsample':         0.989,
    'colsample_bytree':  0.750,
    'reg_alpha':         2.38e-07,
    'reg_lambda':        2.73e-07,
    'min_split_gain':    0.0789,
    'path_smooth':       31.80,
    'w0':                0.177,
}
ANCHOR_REG_OBJECTIVE = 'regression'
ANCHOR_CLF_SPW       = 2.43

# ── TS_REVERSE_SEARCH (LGBM/w0 narrow ±30% + categorical 수동) ──
TS_REVERSE_LOG_KEYS = {
    'learning_rate', 'reg_alpha', 'reg_lambda', 'min_split_gain', 'w0',
}
TS_REVERSE_SEARCH = hpo.narrow_around(
    TS_REVERSE_ANCHOR, log_keys=TS_REVERSE_LOG_KEYS,
    ratio=0.30, int_step_ratio=0.30,
)
# Hard clip — 물리적/논리적 범위 보장
TS_REVERSE_SEARCH['subsample']['low']        = max(0.4, TS_REVERSE_SEARCH['subsample']['low'])
TS_REVERSE_SEARCH['subsample']['high']       = min(1.0, TS_REVERSE_SEARCH['subsample']['high'])
TS_REVERSE_SEARCH['colsample_bytree']['low']  = max(0.1, TS_REVERSE_SEARCH['colsample_bytree']['low'])
TS_REVERSE_SEARCH['colsample_bytree']['high'] = min(1.0, TS_REVERSE_SEARCH['colsample_bytree']['high'])
TS_REVERSE_SEARCH['max_depth']['low']         = max(3,   TS_REVERSE_SEARCH['max_depth']['low'])
TS_REVERSE_SEARCH['num_leaves']['low']        = max(8,   TS_REVERSE_SEARCH['num_leaves']['low'])
TS_REVERSE_SEARCH['min_child_samples']['low'] = max(5,   TS_REVERSE_SEARCH['min_child_samples']['low'])
TS_REVERSE_SEARCH['path_smooth']['low']       = max(0.0, TS_REVERSE_SEARCH['path_smooth']['low'])

# Categorical 수동 추가 (4종 모두 탐색)
TS_REVERSE_SEARCH['reg_objective'] = {
    'type': 'cat',
    'choices': ['regression', 'poisson', 'tweedie_1.2', 'tweedie_1.5'],
}
TS_REVERSE_SEARCH['clf_scale_pos_weight'] = {
    'type': 'cat',
    'choices': [1.0, 1.5, 2.43, 3.5],
}

print(f'EXP_ID={EXP_ID} | USER={USER}')
print(f'N_TRIALS={N_TRIALS} | N_FOLDS={N_FOLDS} | K_INNER={K_INNER} | N_JOBS={N_JOBS}')
print(f'OUT_DIR={OUT_DIR}')
print(f'DB_PATH={DB_PATH}')
print(f'\nTS_REVERSE_SEARCH ({len(TS_REVERSE_SEARCH)} HP):')
for k, spec in sorted(TS_REVERSE_SEARCH.items()):
    if spec['type'] == 'float':
        log_tag = ' log' if spec.get('log') else ''
        print(f'  {k:25s} float [{spec["low"]:.5g}, {spec["high"]:.5g}]{log_tag}')
    elif spec['type'] == 'int':
        print(f'  {k:25s} int   [{spec["low"]}, {spec["high"]}]')
    elif spec['type'] == 'cat':
        print(f'  {k:25s} cat   {spec["choices"]}')

## 3. 데이터 로드 + PP_FIXED 사전 적용 (1회)

PP_FIXED는 trial 내에서 안 흔드므로 study 시작 전 1회만 적용하고 그 결과를 모든 trial에서 공유.

In [ ]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개 샘플')

# PP_FIXED 1회 적용
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PP_FIXED)
xs_train = pp['xs_train']
xs_val   = pp['xs_val']
xs_test  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

# ── 메타피처 추가 (2026-05-09 결정: 트리=position raw + die_xy continuous, numpy 변환 *직전*) ──
feat_cols_clean = add_meta_features(
    xs_train, xs_val, xs_test, feat_cols_clean,
    position_mode='raw', use_die_xy=True,
)

X_train = xs_train[feat_cols_clean].values.astype(np.float64)
X_val   = xs_val[feat_cols_clean].values.astype(np.float64)
X_test  = xs_test[feat_cols_clean].values.astype(np.float64)

uid_train_die = xs_train[KEY_COL].values
uid_val_die   = xs_val[KEY_COL].values
uid_test_die  = xs_test[KEY_COL].values

y_train_unit_s = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit_s   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit_s  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

# die-level broadcast (Path B는 die 단위 학습)
y_train_die_broadcast = pd.Series(uid_train_die).map(y_train_unit_s).values.astype(np.float64)
assert not pd.isna(y_train_die_broadcast).any(), 'unmapped train die y'
y_bin_die_broadcast = (y_train_die_broadcast > 0).astype(np.int32)

n_train_die = len(X_train)
n_val_die   = len(X_val)
n_test_die  = len(X_test)

print(f'[전처리 완료] feat_cols: {len(feat_cols_clean)}')
print(f'  X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}')
print(f'  unit train={len(y_train_unit_s):,}, val={len(y_val_unit_s):,}, test={len(y_test_unit_s):,}')
print(f'  y_bin (broadcasted y>0) pos ratio: {y_bin_die_broadcast.mean():.4f}')

## 4. K-fold split + helper + Optuna objective

- KFold는 **unit ID 단위 분할** (strategy_common §6) — 같은 unit의 4 die는 같은 fold
- helper inline: `_build_reg_params`, `_train_path_b(mode='innerOOF')`, `_mean_die_to_unit`, `_rmse_unit`
- inner KFold도 반드시 **unit 단위 분할** (die-level 분할 시 leakage)
- pruning: MedianPruner(n_warmup_steps=2)

In [ ]:
unique_units = y_train_unit_s.index.values
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf.split(unique_units))


def _build_reg_params(hp, reg_obj):
    p = dict(hp)
    if reg_obj.startswith('tweedie'):
        p['objective'] = 'tweedie'
        p['tweedie_variance_power'] = float(reg_obj.split('_')[1])
    else:
        p['objective'] = reg_obj
    return p


def _train_path_b(X_tr, y_tr_continuous, y_tr_bin, X_others,
                  hp, w0, reg_obj, clf_spw,
                  uid_tr, k_inner=5, seed=42):
    """Path B 1 outer fold 학습 + 예측 (inner-OOF 모드 고정).

    - Stage 1 reg_full: outer-train 전체 학습 (추론용)
    - Stage 2 clf 학습 feature: outer-train 안에서 inner KFold(unit-level) OOF reg 예측
    - 추론: 모든 X_others에 reg_full → clf 순으로 적용
    """
    sw_full = np.where(y_tr_continuous == 0, w0, 1.0)
    y_tr_log = y_tr_continuous  # ★ strategy_common §24 — 트리 target_transform=none (변수명 유지, log 의미 없음)
    reg_params = _build_reg_params(hp, reg_obj)

    # ── Stage 1 reg_full (추론용) ──
    reg_full = lgb.LGBMRegressor(**reg_params)
    reg_full.fit(X_tr, y_tr_log, sample_weight=sw_full)

    # ── Stage 2 학습 feature: inner-OOF reg 예측 ──
    unique_inner_units = np.unique(uid_tr)
    inner_kf = KFold(n_splits=k_inner, shuffle=True, random_state=seed)
    reg_train_oof_log = np.full(len(X_tr), np.nan)
    for itr_uidx, ivl_uidx in inner_kf.split(unique_inner_units):
        itr_units = unique_inner_units[itr_uidx]
        ivl_units = unique_inner_units[ivl_uidx]
        itr_die_mask = np.isin(uid_tr, itr_units)
        ivl_die_mask = np.isin(uid_tr, ivl_units)
        sw_inner = np.where(y_tr_continuous[itr_die_mask] == 0, w0, 1.0)
        reg_inner = lgb.LGBMRegressor(**reg_params)
        reg_inner.fit(
            X_tr[itr_die_mask],
            y_tr_log[itr_die_mask],
            sample_weight=sw_inner,
        )
        reg_train_oof_log[ivl_die_mask] = reg_inner.predict(X_tr[ivl_die_mask])
    assert not np.isnan(reg_train_oof_log).any(), 'inner OOF coverage bug'
    reg_train_y_for_clf = np.clip(reg_train_oof_log, 0.0, None)  # ★ §24 — log space 아님
    X_tr_aug = np.hstack([X_tr, reg_train_y_for_clf.reshape(-1, 1)])

    # ── Stage 2 clf ──
    clf_params = dict(hp)
    clf_params['objective'] = 'binary'
    clf_params['scale_pos_weight'] = clf_spw
    clf = lgb.LGBMClassifier(**clf_params)
    clf.fit(X_tr_aug, y_tr_bin)

    # ── 추론 (X_others 모두 reg_full + clf) ──
    results = []
    for X_o in X_others:
        reg_log_o = reg_full.predict(X_o)
        reg_y_o   = np.clip(reg_log_o, 0.0, None)  # ★ §24 — log space 아님
        X_o_aug   = np.hstack([X_o, reg_y_o.reshape(-1, 1)])
        prob_o = np.clip(clf.predict_proba(X_o_aug)[:, 1], 0.0, 1.0)
        final_o = prob_o * reg_y_o
        results.append((prob_o, reg_y_o, final_o))
    return results, (reg_full, clf)


def _mean_die_to_unit(pred_die, uid_die):
    df = pd.DataFrame({KEY_COL: uid_die, 'pred': pred_die})
    return df.groupby(KEY_COL, sort=False)['pred'].mean().reset_index()


def objective(trial):
    t0 = time.time()
    sampled = hpo.sample_from_space(trial, TS_REVERSE_SEARCH)
    w0       = sampled.pop('w0')
    reg_obj  = sampled.pop('reg_objective')
    clf_spw  = sampled.pop('clf_scale_pos_weight')
    hp = sampled  # 나머지 LGBM HP만 남음

    hp['random_state']   = SEED
    hp['n_jobs']         = N_JOBS
    hp['verbose']        = -1
    hp['subsample_freq'] = 1

    fold_oof_rmse = []
    oof_pred_unit = pd.Series(np.nan, index=y_train_unit_s.index, dtype=np.float64)

    for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
        tr_units = unique_units[tr_uidx]
        vl_units = unique_units[vl_uidx]
        tr_mask = np.isin(uid_train_die, tr_units)
        vl_mask = np.isin(uid_train_die, vl_units)

        results, _ = _train_path_b(
            X_train[tr_mask], y_train_die_broadcast[tr_mask], y_bin_die_broadcast[tr_mask],
            [X_train[vl_mask]],
            hp, w0, reg_obj, clf_spw,
            uid_tr=uid_train_die[tr_mask], k_inner=K_INNER, seed=SEED,
        )
        _, _, f_vl = results[0]
        unit_pred_df = _mean_die_to_unit(f_vl, uid_train_die[vl_mask])

        oof_pred_unit.loc[unit_pred_df[KEY_COL].values] = unit_pred_df['pred'].values
        y_vl = y_train_unit_s.loc[unit_pred_df[KEY_COL].values].values
        fold_rmse = float(np.sqrt(np.mean((unit_pred_df['pred'].values - y_vl) ** 2)))
        fold_oof_rmse.append(fold_rmse)

        avg = float(np.mean(fold_oof_rmse))
        trial.report(avg, step=fold_idx)
        if trial.should_prune():
            trial.set_user_attr('pruned_at_fold', fold_idx + 1)
            trial.set_user_attr('elapsed_sec', time.time() - t0)
            trial.set_user_attr('w0', w0)
            trial.set_user_attr('reg_objective', reg_obj)
            trial.set_user_attr('clf_scale_pos_weight', clf_spw)
            raise optuna.TrialPruned()

    if oof_pred_unit.isna().any():
        raise RuntimeError('OOF NaN — fold 누락')

    oof_rmse = float(np.sqrt(np.mean((oof_pred_unit.values - y_train_unit_s.values) ** 2)))
    elapsed = time.time() - t0
    trial.set_user_attr('elapsed_sec', elapsed)
    trial.set_user_attr('w0', w0)
    trial.set_user_attr('reg_objective', reg_obj)
    trial.set_user_attr('clf_scale_pos_weight', clf_spw)
    trial.set_user_attr('fold_oof_rmse', fold_oof_rmse)
    print(f'  trial #{trial.number}: oof={oof_rmse:.6f}, w0={w0:.3f}, reg={reg_obj}, spw={clf_spw}, elapsed={elapsed:.0f}s')
    return oof_rmse


print(f'fold split: {N_FOLDS} folds, unit 단위 분할, seed={SEED}')
print(f'inner KFold: K_INNER={K_INNER}, unit 단위')

## 5. Optuna study 생성 + anchor enqueue + optimize

- TPESampler(seed=None, multivariate=True, group=True) — strategy_common §4
- `enqueue_anchor`로 첫 trial은 1차 best HP 그대로

In [ ]:
sampler = TPESampler(
    seed=None,
    multivariate=True,
    group=True,
    n_startup_trials=N_STARTUP_TRIALS,
)
pruner = MedianPruner(n_startup_trials=N_STARTUP_TRIALS, n_warmup_steps=2)

study = optuna.create_study(
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    sampler=sampler,
    pruner=pruner,
    direction='minimize',
    load_if_exists=True,
)

# anchor 첫 trial 강제 (strategy_common §5)
ANCHOR_FOR_ENQUEUE = dict(TS_REVERSE_ANCHOR)
ANCHOR_FOR_ENQUEUE['reg_objective']        = ANCHOR_REG_OBJECTIVE
ANCHOR_FOR_ENQUEUE['clf_scale_pos_weight'] = ANCHOR_CLF_SPW
if len(study.trials) == 0:
    hpo.enqueue_anchor(study, ANCHOR_FOR_ENQUEUE)
else:
    print(f'[enqueue skip] 기존 trial {len(study.trials)} 있음 — resume')

study_meta = {
    'exp_id': EXP_ID, 'user': USER, 'model': 'Reverse Two-Stage (path B, innerOOF)',
    'n_trials': N_TRIALS, 'n_folds': N_FOLDS, 'k_inner': K_INNER, 'n_jobs': N_JOBS,
    'pp_fixed': PP_FIXED,
    'anchor': TS_REVERSE_ANCHOR,
    'anchor_reg_objective': ANCHOR_REG_OBJECTIVE,
    'anchor_clf_spw': ANCHOR_CLF_SPW,
    'sampler': 'TPE seed=None multivariate group',
    'pruner':  f'MedianPruner n_startup={N_STARTUP_TRIALS} n_warmup=2',
    'CLIP_Y_EXTREME': CLIP_Y_EXTREME, 'SEED': int(SEED),
}
for k, v in study_meta.items():
    study.set_user_attr(k, str(v))

print(f'study: {study.study_name}, DB: {DB_PATH}')
print(f'기존 trial: {len(study.trials)}')

t_start = time.time()
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SEC, n_jobs=1, show_progress_bar=True)
print(f'\n[HPO 완료] 전체 {time.time()-t_start:.0f}s, total trials={len(study.trials)}')
print(f'  best OOF RMSE: {study.best_value:.6f}')


## 6. Best trial 정보 + anchor enqueue 검증

In [ ]:
best_trial = study.best_trial
best_params_full = best_trial.params  # LGBM HP + w0 + reg_objective + clf_scale_pos_weight 포함
best_w0       = best_params_full['w0']
best_reg_obj  = best_params_full['reg_objective']
best_clf_spw  = best_params_full['clf_scale_pos_weight']
hp_best = {
    k: v for k, v in best_params_full.items()
    if k not in ['w0', 'reg_objective', 'clf_scale_pos_weight']
}

print(f'=== Best Trial #{best_trial.number} ===')
print(f'  OOF RMSE      : {best_trial.value:.6f}')
print(f'  best w0       : {best_w0:.4f}')
print(f'  best reg_obj  : {best_reg_obj}')
print(f'  best clf_spw  : {best_clf_spw}')
print(f'  elapsed       : {best_trial.user_attrs.get("elapsed_sec", 0):.0f}s')
for k, v in sorted(hp_best.items()):
    print(f'    {k}: {v}')

# enqueue 검증 — trial 0이 anchor와 일치?
trial0 = study.trials[0]
def _eq(a, b, tol=1e-9):
    if isinstance(a, str) or isinstance(b, str):
        return a == b
    try:
        return abs(float(a) - float(b)) < tol
    except (TypeError, ValueError):
        return a == b
anchor_check = all(
    _eq(trial0.params.get(k), v) for k, v in ANCHOR_FOR_ENQUEUE.items()
)
print(f'\n[검증] trial 0 == anchor? {anchor_check}')

## 7. Best HP 5-fold refit (innerOOF) + die-level prob/reg/pred 캐쳐

In [ ]:
hp_refit = dict(hp_best)
hp_refit['random_state']   = SEED
hp_refit['n_jobs']         = N_JOBS
hp_refit['verbose']        = -1
hp_refit['subsample_freq'] = 1

oof_die_prob   = np.full(n_train_die, np.nan)
oof_die_reg    = np.full(n_train_die, np.nan)
oof_die_pred   = np.full(n_train_die, np.nan)

val_die_prob   = np.zeros(n_val_die)
val_die_reg    = np.zeros(n_val_die)
val_die_pred   = np.zeros(n_val_die)
test_die_prob  = np.zeros(n_test_die)
test_die_reg   = np.zeros(n_test_die)
test_die_pred  = np.zeros(n_test_die)

fold_models = []  # [(reg_full, clf), ...] per fold

print(f'=== Best HP 5-fold refit (Path B + innerOOF) ===')
t0 = time.time()
for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
    tr_units = unique_units[tr_uidx]
    vl_units = unique_units[vl_uidx]
    tr_mask = np.isin(uid_train_die, tr_units)
    vl_mask = np.isin(uid_train_die, vl_units)

    results, models = _train_path_b(
        X_train[tr_mask], y_train_die_broadcast[tr_mask], y_bin_die_broadcast[tr_mask],
        [X_train[vl_mask], X_val, X_test],
        hp_refit, best_w0, best_reg_obj, best_clf_spw,
        uid_tr=uid_train_die[tr_mask], k_inner=K_INNER, seed=SEED,
    )
    (p_vl, r_vl, f_vl), (p_v, r_v, f_v), (p_t, r_t, f_t) = results

    oof_die_prob[vl_mask] = p_vl
    oof_die_reg[vl_mask]  = r_vl
    oof_die_pred[vl_mask] = f_vl

    val_die_prob  += p_v / N_FOLDS
    val_die_reg   += r_v / N_FOLDS
    val_die_pred  += f_v / N_FOLDS
    test_die_prob += p_t / N_FOLDS
    test_die_reg  += r_t / N_FOLDS
    test_die_pred += f_t / N_FOLDS

    fold_models.append(models)
    print(f'  fold {fold_idx+1}/{N_FOLDS} done ({time.time()-t0:.0f}s)')

assert not np.isnan(oof_die_prob).any()
assert not np.isnan(oof_die_reg).any()
assert not np.isnan(oof_die_pred).any()
print(f'\n[refit 완료] die-level prob/reg/pred 캐쳐 OK')

## 8. 후처리 — 집계 8 + position Optuna + π threshold + zero_clip(log)

- 분류 threshold (§9): **APPLY** — Reverse는 die-level prob을 cut 안 했으므로 후처리에서 unit 평균 prob에 threshold 탐색
- 집계 다양성 (§10): 8종
- Position 가중치 (§11): Optuna sub-study 50 trial
- zero_clip (§12): original space 비교 (TARGET_TRANSFORM='none', strategy_common §24)

In [ ]:
pp_res = postprocess.tune_and_apply(
    xs_train, xs_val, xs_test,
    die_pred_train=oof_die_pred,
    die_pred_val=val_die_pred,
    die_pred_test=test_die_pred,
    die_pi_train=oof_die_prob,
    die_pi_val=val_die_prob,
    die_pi_test=test_die_prob,
    y_train_unit=ys_input['train'],
    use_pi_threshold=True,                 # Reverse는 die-level prob cut 안 함 → 후처리에서 탐색
    agg_methods=postprocess.AGG_METHODS,   # 8종
    zero_clip_log_space=False,             # ★ strategy_common §24 — 트리 target_transform=none, 학습 공간 original
    position_method='optuna',
    position_optuna_n_trials=50,
)

print(f'\n[Postprocess]')
print(f'  best_agg            : {pp_res["best_agg"]}')
print(f'  pos_weights         : {pp_res["pos_weights"]}')
print(f'  best_pi_threshold   : {pp_res["best_pi_threshold"]}')
print(f'  best_zero_clip(log) : {pp_res["best_zero_clip"]:.4f}')
print(f'  position_method     : {pp_res["position_method"]}')
print(f'  train_rmse          : {pp_res["train_rmse"]:.6f}')

# ── val/test 후처리 RMSE (마이그레이션 후 누락 — 복원) ──
if pp_res.get('final_val_unit') is not None:
    _val_pred = pp_res['final_val_unit'].set_index(KEY_COL)['pred'].loc[y_val_unit_s.index]
    val_rmse  = float(np.sqrt(np.mean((_val_pred.values  - y_val_unit_s.values)  ** 2)))
    print(f'  val_rmse            : {val_rmse:.6f}')
if pp_res.get('final_test_unit') is not None:
    _test_pred = pp_res['final_test_unit'].set_index(KEY_COL)['pred'].loc[y_test_unit_s.index]
    test_rmse  = float(np.sqrt(np.mean((_test_pred.values - y_test_unit_s.values) ** 2)))
    print(f'  test_rmse           : {test_rmse:.6f}')

print(f'  agg_rmses           : {pp_res["agg_rmses"]}')

## 9. 산출물 9개 저장 (strategy_common §15)

best_params.json + fold_models.pkl + 6 CSV (die ×3 + unit ×3) + optuna_*.db

In [ ]:
import json, pickle, hashlib

# 1) fold_models.pkl — [(reg_full, clf), ...] per fold
with open(os.path.join(OUT_DIR, 'fold_models.pkl'), 'wb') as f:
    pickle.dump({
        'fold_models':   fold_models,    # list of (reg, clf)
        'feature_names': feat_cols_clean,
        'model_name':    'ts_reverse',
        'n_folds':       N_FOLDS,
    }, f)

# 2) best_params.json
uid_arr = ys_input['train'][KEY_COL].unique()
unit_ids_hash = hashlib.sha1(','.join(map(str, uid_arr)).encode()).hexdigest()

best_meta = {
    'exp_id':                EXP_ID,
    'model_name':            'ts_reverse',
    'best_trial_number':     best_trial.number,
    'best_oof_rmse':         float(best_trial.value),
    'best_params_resolved':  hp_refit,
    'best_w0':               float(best_w0),
    'best_reg_objective':    best_reg_obj,
    'best_clf_scale_pos_weight': float(best_clf_spw),
    'feature_names':         feat_cols_clean,
    'n_features':            len(feat_cols_clean),
    'n_folds':               N_FOLDS,
    'k_inner':               K_INNER,
    'unit_ids_hash':         unit_ids_hash,
    'n_units_train':         int(len(uid_arr)),
    'effective_pp_params':   PP_FIXED,
    'study_meta':            study_meta,
    'postprocess': {
        'best_agg':            pp_res['best_agg'],
        'pos_weights':         pp_res['pos_weights'].tolist() if pp_res['pos_weights'] is not None else None,
        'best_pi_threshold':   (float(pp_res['best_pi_threshold'])
                                if pp_res['best_pi_threshold'] is not None else None),
        'best_zero_clip':      float(pp_res['best_zero_clip']),
        'zero_clip_log_space': pp_res['zero_clip_log_space'],
        'position_method':     pp_res['position_method'],
        'agg_rmses':           {k: float(v) for k, v in pp_res['agg_rmses'].items()},
        'train_rmse':          float(pp_res['train_rmse']),
    },
}
with open(os.path.join(OUT_DIR, 'best_params.json'), 'w', encoding='utf-8') as f:
    json.dump(best_meta, f, indent=2, ensure_ascii=False, default=str)

# 3-5) die-level CSV
def _build_die_df(uid, die_id, position, prob, reg, pred, y_unit):
    df = pd.DataFrame({
        KEY_COL: uid, DIE_KEY_COL: die_id, 'position': position,
        'prob': prob, 'reg': reg, 'pred': pred,
    })
    if y_unit is not None:
        df[TARGET_COL] = df[KEY_COL].map(y_unit)
    return df

_build_die_df(
    uid_train_die, xs_train[DIE_KEY_COL].values, xs_train['position'].values,
    oof_die_prob, oof_die_reg, oof_die_pred, y_train_unit_s,
).to_csv(os.path.join(OUT_DIR, 'oof_die.csv'), index=False)
_build_die_df(
    uid_val_die, xs_val[DIE_KEY_COL].values, xs_val['position'].values,
    val_die_prob, val_die_reg, val_die_pred, y_val_unit_s,
).to_csv(os.path.join(OUT_DIR, 'val_die.csv'), index=False)
_build_die_df(
    uid_test_die, xs_test[DIE_KEY_COL].values, xs_test['position'].values,
    test_die_prob, test_die_reg, test_die_pred, y_test_unit_s,
).to_csv(os.path.join(OUT_DIR, 'test_die.csv'), index=False)

# 6-8) unit-level CSV (postprocess 적용본 + health merge)
def _build_unit_df(unit_pred_df, y_unit):
    out = unit_pred_df.copy()
    out[TARGET_COL] = out[KEY_COL].map(y_unit)
    return out

_build_unit_df(pp_res['final_train_unit'], y_train_unit_s).to_csv(os.path.join(OUT_DIR, 'oof_unit.csv'),  index=False)
_build_unit_df(pp_res['final_val_unit'],   y_val_unit_s  ).to_csv(os.path.join(OUT_DIR, 'val_unit.csv'),  index=False)
_build_unit_df(pp_res['final_test_unit'],  y_test_unit_s ).to_csv(os.path.join(OUT_DIR, 'test_unit.csv'), index=False)

# 9) optuna_*.db는 study.optimize에서 자동 저장

# 산출물 확인
print(f'\n저장 완료: {OUT_DIR}')
for fn in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, fn)) / 1024
    print(f'  {fn:30s}  {sz:10,.1f} KB')

# Colab → 로컬 zip 다운로드
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip = shutil.make_archive(os.path.join('/content', f'ts_reverse_{EXP_ID}_outputs'), 'zip', OUT_DIR)
    print(f'\n[zip 생성] {_zip} ({os.path.getsize(_zip)/1024:.1f} KB)')
    try:
        files.download(_zip)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크 클릭')
        display(FileLink(_zip))
except ImportError:
    pass
